[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/20_weight_init_solution.ipynb)

# 🟢 Solution: Kaiming Initialization（参考版）

## 📋 题目描述

**难度：** Easy

实现 Kaiming（He）权重初始化。

Kaiming 初始化根据 fan-in 设置权重方差，以在 ReLU 网络中保持信号幅度，防止激活值消失或爆炸。

**签名:** `kaiming_init(weight) -> Tensor`

**参数:**
- `weight` — 需要原地初始化的张量 (out_features, in_features)

**返回:** 同一张量（原地操作）

**约束:**
- `std = sqrt(2 / fan_in)`，其中 `fan_in = weight.shape[1]`
- 用 `normal(0, std)` 填充
- 更小的 fan_in 应产生更大的 std

**提示：** `fan_in = weight.shape[1]` → `std = sqrt(2 / fan_in)`
`weight.normal_(0, std)` 原地操作 → 返回 `weight`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

import math

def kaiming_init(weight):
    # ---- Step 1: 获取 fan-in ----
    # fan_in = 每个输出神经元连接的输入神经元数量
    # 对于形状 (out_features, in_features) 的权重矩阵
    # fan_in = in_features = weight.shape[1]
    # 例如 Linear(256, 512) 的权重 shape (512, 256)，fan_in = 256
    fan_in = weight.shape[1] if weight.dim() >= 2 else weight.shape[0]

    # ---- Step 2: 计算标准差 ----
    # Kaiming 公式：std = sqrt(2 / fan_in)
    # 为什么是 2/fan_in？
    #   假设输入 x 的方差为 Var(x)，权重 w 的方差为 Var(w)
    #   线性层输出 y = Σ(w_i * x_i)
    #   Var(y) = fan_in * Var(w) * Var(x)（独立随机变量乘积的方差）
    #   要让 Var(y) = Var(x)（信号不放大也不缩小），需要：
    #   fan_in * Var(w) = 1 → Var(w) = 1/fan_in → std = 1/sqrt(fan_in)
    #   但 ReLU 会丢弃一半神经元（负值变零），所以方差减半
    #   补偿：Var(w) = 2/fan_in → std = sqrt(2/fan_in)
    std = math.sqrt(2.0 / fan_in)

    # ---- Step 3: 正态分布填充 ----
    # normal_(0, std) 原地操作，用 N(0, std²) 填充
    # torch.no_grad() 确保初始化不被 autograd 追踪
    with torch.no_grad():
        weight.normal_(0, std)
    return weight


In [ ]:
w = torch.empty(512, 256)
kaiming_init(w)
print(f'Shape: {w.shape}, Std: {w.std():.4f}, Expected: {math.sqrt(2/256):.4f}')


In [ ]:
from torch_judge import check
check('weight_init')


## 📝 核心思路总结

1. **Kaiming 公式**：`std = sqrt(2 / fan_in)`，为 ReLU 网络量身定制
2. **推导关键**：ReLU 丢弃一半神经元 → 方差减半 → 需要 `2/fan_in` 补偿
3. **原地操作**：`weight.normal_(0, std)` 在 `no_grad()` 下执行
4. **vs Xavier**：Xavier 用 `2/(fan_in+fan_out)`，适合无 ReLU 的网络
